In [28]:
## Imports
import numpy as np
import glob
import matplotlib.pyplot as plt
import os
import fitsio as fio
import pandas as pd
import warnings

from pathlib import Path
from astropy import units as u
from astropy.coordinates import SkyCoord, Angle, match_coordinates_sky
from astropy.io import fits
from astropy.table import Table

## Making the cross correlations 
from pycorr import (
    TwoPointCorrelationFunction, TwoPointEstimator, NaturalTwoPointEstimator, utils, setup_logging
)
setup_logging()

USER_ROOT = Path('/global/cfs/cdirs/desicollab/users/jchdj')
DATA_ROOT = USER_ROOT / 'desi-y3-hsc' / 'data'
DESI_ROOT = DATA_ROOT / 'desi' / 'cat'
HSC_ROOT = DATA_ROOT / 'hsc' / 'cat'
XMATCH_ROOT = DATA_ROOT / 'xmatch'

desi_file = Path(XMATCH_ROOT, 'desi_hsc.fits')
hsc_file = Path(HSC_ROOT, 'hscy3_cat.fits')
print(f'HSC catalog : {hsc_file}')
print(f'DESI catalog on HSC footprint : {desi_file}')
assert desi_file.is_file(), f'DESI catalog not found at {desi_file}'
assert hsc_file.is_file(), f'HSC catalog not found at {hsc_file}'

HSC catalog : /global/cfs/cdirs/desicollab/users/jchdj/desi-y3-hsc/data/hsc/cat/hscy3_cat.fits
DESI catalog on HSC footprint : /global/cfs/cdirs/desicollab/users/jchdj/desi-y3-hsc/data/xmatch/desi_hsc.fits


In [7]:
desi = fio.read(desi_file)
hsc = fio.read(hsc_file)

In [34]:
# Parameters
# bins : 0 to 200 Mpc/h, 50 bins (4 Mpc/h separation)
bin_edges = np.linspace(0., 200., 51)
print(f'bin edges : {bin_edges[:5]} ... {bin_edges[-5:]}')

bin edges : [ 0.  4.  8. 12. 16.] ... [184. 188. 192. 196. 200.]


In [ ]:
dp1_desi = [desi['TARGET_RA'][0::1000], desi['TARGET_DEC'][0::1000]]
dp2_hsc = [hsc['RA'][0::1000], hsc['Dec'][0::1000]]

w2_hsc = hsc['weight'][0::1000]

In [38]:
from desitarget.targetmask import desi_mask

desi_tgt = desi['DESI_TARGET']

is_bgs  = (desi_tgt & desi_mask.BGS_ANY != 0)   #- instead of 2**60
is_lrg  = (desi_tgt & desi_mask.LRG != 0)
is_elg  = (desi_tgt & desi_mask.ELG != 0)
is_qso  = (desi_tgt & desi_mask.QSO != 0)
is_mws  = (desi_tgt & desi_mask.MWS_ANY != 0)
is_scnd = (desi_tgt & desi_mask.SCND_ANY != 0)
masks = {
    'BGS': is_bgs,
    'LRG': is_lrg,
    'ELG': is_elg,
    'QSO': is_qso,
    'MWS': is_mws,
    'SCND': is_scnd
}
bins_bgs = np.linspace(0, 0.7, 8) # 0 < z < 0.6
bins_lrg = np.linspace(0.3, 1.1, 9) # 0.4 < z < 1
bins_elg = np.linspace(0.6, 1.7, 12) # 0.6 < z < 1.6
bins_qso = np.linspace(0.9, 2.2, 14) # 0.9 < z < 2.1
print(f'bins_bgs : {bins_bgs}')
print(f'bins_lrg : {bins_lrg}')
print(f'bins_elg : {bins_elg}')
print(f'bins_qso : {bins_qso}')

bins_bgs : [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7]
bins_lrg : [0.3 0.4 0.5 0.6 0.7 0.8 0.9 1.  1.1]
bins_elg : [0.6 0.7 0.8 0.9 1.  1.1 1.2 1.3 1.4 1.5 1.6 1.7]
bins_qso : [0.9 1.  1.1 1.2 1.3 1.4 1.5 1.6 1.7 1.8 1.9 2.  2.1 2.2]


In [ ]:
# pre-select on redshift first
# target class + 0.1 redshift bins but angular separation
tpcf = TwoPointCorrelationFunction(
    edges=bin_edges,
    data_positions1=dp1_desi,
    data_positions2=dp2_hsc,
    randoms_positions1=dp1_desi,
    randoms_positions2=dp2_hsc,
    data_weights1=w1_desi,
    data_weights2=w2_hsc,
    mode='theta',
    position_type='rd', # 'rd' for RA/Dec
    engine='corrfunc',
    estimator='landyszalay',
    )

[002598.58]  03-24 15:21  TwoPointCorrelationFunction  INFO     Using estimator <class 'pycorr.twopoint_estimator.LandySzalayTwoPointEstimator'>.
[002598.59]  03-24 15:21  TwoPointCorrelationFunction  INFO     Running cross-correlation.
[002598.59]  03-24 15:21  TwoPointCorrelationFunction  INFO     Computing two-point counts D1D2.
[002600.06]  03-24 15:21  TwoPointCorrelationFunction  INFO     Computing two-point counts D1R2.
[002601.53]  03-24 15:21  TwoPointCorrelationFunction  INFO     Computing two-point counts R1D2.
[002602.97]  03-24 15:21  TwoPointCorrelationFunction  INFO     Computing two-point counts R1R2.
[002604.32]  03-24 15:21  TwoPointCorrelationFunction  INFO     Correlation function computed in elapsed time 5.74 s.


In [ ]:
tpcf.

In [15]:
import os
os.listdir('/global/cfs/projectdirs/desi/science/c3/DESI-Lensing/lensingsurvey_catalogues/')

['makecats_nersc.py',
 'kids',
 'sdss',
 '.ipynb_checkpoints',
 'hscy3',
 'boss',
 'hsc',
 'desy1',
 'cut_catalogues',
 'desy3']

In [13]:
import fitsio as fio
tbl = fio.FITS('/global/cfs/projectdirs/desi/science/c3/DESI-Lensing/lensingsurvey_catalogues/cut_catalogues/hscy3/hscy3_cat_ELG.fits')

In [14]:
tbl[1].get_nrows()

23302186

In [ ]:
os.listdir('/pscratch/sd/x/xiangchl/data/catalog/')

['GAMA15H_star.fits',
 'WIDE12H_pz_2.fits',
 'XMM_pz_2.fits',
 'HECTOMAP_pz_2.fits',
 'GAMA09H_star.fits',
 'XMM_star.fits',
 'GAMA15H_pz_2.fits',
 'WIDE12H_star.fits',
 'VVDS_pz_2.fits',
 'HECTOMAP_star.fits',
 'psf_systematics',
 'VVDS_star.fits',
 'GAMA09H_pz_2.fits']

In [24]:
import glob
from pathlib import Path
fs = glob.glob('/global/cfs/cdirs/desicollab/science/c3/DESI-Lensing/desi_catalogues/weight_prob_obs/hscy3/*')
print([Path(f).name for f in fs])

['ELG_LOPnotqso_1_clustering.ran.fits', 'BGS_BRIGHT_3_clustering.ran.fits', 'LRG+ELG_LOPnotqso_NGC_1_clustering.ran.fits', 'BGS_BRIGHT_SGC_1_clustering.ran.fits', 'ELG_LOPnotqso_SGC_0_clustering.ran.fits', 'ELG_LOPnotqso_NGC_2_clustering.ran.fits', 'LRG+ELG_LOPnotqso_SGC_0_clustering.ran.fits', 'LRG+ELG_LOPnotqso_SGC_3_clustering.ran.fits', 'LRG_NGC_3_clustering.ran.fits', 'LRG_NGC_1_clustering.ran.fits', 'ELG_LOPnotqso_SGC_3_clustering.ran.fits', 'LRG_clustering.dat.fits', 'BGS_BRIGHT_0_clustering.ran.fits', 'ELG_LOPnotqso_2_clustering.ran.fits', 'LRG_3_clustering.ran.fits', 'ELG_LOPnotqso_3_clustering.ran.fits', 'LRG_2_clustering.ran.fits', 'BGS_BRIGHT_1_clustering.ran.fits', 'LRG_1_clustering.ran.fits', 'BGS_BRIGHT_2_clustering.ran.fits', 'ELG_LOPnotqso_clustering.dat.fits', 'LRG_0_clustering.ran.fits', 'ELG_LOPnotqso_0_clustering.ran.fits', 'BGS_BRIGHT_clustering.dat.fits']


In [25]:
hsc_ran_root = '/global/cfs/cdirs/desicollab/science/c3/DESI-Lensing/desi_catalogues/weight_prob_obs/hscy3/'
tbl = fio.FITS(f'{hsc_ran_root}ELG_LOPnotqso_1_clustering.ran.fits')

In [27]:
tbl[1].get_colnames()

['TARGETID',
 'RA',
 'DEC',
 'NTILE',
 'PHOTSYS',
 'FRAC_TLOBS_TILES',
 'Z',
 'WEIGHT',
 'WEIGHT_SYS',
 'WEIGHT_COMP',
 'WEIGHT_ZFAIL',
 'WEIGHT_SN',
 'WEIGHT_RF',
 'TARGETID_DATA']

In [ ]:
# Search for files with "ran" in their names in the specified directories
search_dirs = [
    '/pscratch/sd/x/xiangchl/data/catalog/',
    '/global/cfs/cdirs/desicollab/science/c3/DESI-Lensing/'
]



print(f"Found {len(ran_files)} files containing 'ran' in their names:")
for file in ran_files[:20]:  # Limit output to first 20 files if there are many
    print(file)

if len(ran_files) > 20:
    print(f"... and {len(ran_files) - 20} more files")

Found 4 files containing 'ran' in their names:
/global/cfs/projectdirs/desi/science/c3/DESI-Lensing/lensingsurvey_catalogues/boss/boss_random_cat.dat
/global/cfs/projectdirs/desi/science/c3/DESI-Lensing/lensingsurvey_catalogues/boss/boss_random_cat.fits
/global/cfs/projectdirs/desi/science/c3/DESI-Lensing/lensingsurvey_catalogues/boss/raw/random0_DR12v5_CMASSLOWZTOT_North.fits
/global/cfs/projectdirs/desi/science/c3/DESI-Lensing/lensingsurvey_catalogues/boss/raw/random0_DR12v5_CMASSLOWZTOT_South.fits


In [ ]:
os.listdir('/global/cfs/cdirs/desicollab/science/c3/DESI-Lensing/prelim_hscy3/gfarm.ipmu.jp/~surhud/S19ACatalogs/catalog_tracts/VVDS_tracts')

In [29]:
tbl = fio.FITS('/global/cfs/cdirs/desicollab/science/c3/DESI-Lensing/desi_catalogues/weight_prob_obs/hscy3/LRG_clustering.dat.fits')

In [30]:
tbl[1].get_colnames()

['TARGETID',
 'Z',
 'NTILE',
 'RA',
 'DEC',
 'PHOTSYS',
 'FRAC_TLOBS_TILES',
 'WEIGHT_ZFAIL',
 'BITWEIGHTS',
 'PROB_OBS',
 'WEIGHT_SYS',
 'WEIGHT',
 'WEIGHT_COMP']

In [5]:
import numpy as np
bins_qso = np.arange(0.9, 2.2, 0.1)
print(bins_qso)

[0.9 1.  1.1 1.2 1.3 1.4 1.5 1.6 1.7 1.8 1.9 2.  2.1 2.2]


In [6]:
import fitsio as fio
import glob
root = '/global/cfs/projectdirs/desi/survey/catalogs/Y3/LSS/loa-v1/LSScats/v1.1/nonKP'
files = glob.glob(f'{root}/*dat.fits')
print(files)

['/global/cfs/projectdirs/desi/survey/catalogs/Y3/LSS/loa-v1/LSScats/v1.1/nonKP/ELGnotqso_SGC_clustering.dat.fits', '/global/cfs/projectdirs/desi/survey/catalogs/Y3/LSS/loa-v1/LSScats/v1.1/nonKP/BGS_ANY_NGC_clustering.dat.fits', '/global/cfs/projectdirs/desi/survey/catalogs/Y3/LSS/loa-v1/LSScats/v1.1/nonKP/BGS_BRIGHT-21.5_NGC_clustering.dat.fits', '/global/cfs/projectdirs/desi/survey/catalogs/Y3/LSS/loa-v1/LSScats/v1.1/nonKP/ELGnotqso_clustering.dat.fits', '/global/cfs/projectdirs/desi/survey/catalogs/Y3/LSS/loa-v1/LSScats/v1.1/nonKP/BGS_ANY_clustering.dat.fits', '/global/cfs/projectdirs/desi/survey/catalogs/Y3/LSS/loa-v1/LSScats/v1.1/nonKP/BGS_BRIGHT_SGC_clustering.dat.fits', '/global/cfs/projectdirs/desi/survey/catalogs/Y3/LSS/loa-v1/LSScats/v1.1/nonKP/BGS_BRIGHT-20.2_clustering.dat.fits', '/global/cfs/projectdirs/desi/survey/catalogs/Y3/LSS/loa-v1/LSScats/v1.1/nonKP/LRG_NGC_clustering.dat.fits', '/global/cfs/projectdirs/desi/survey/catalogs/Y3/LSS/loa-v1/LSScats/v1.1/nonKP/LRG_clust

In [8]:
tbl = fio.FITS(files[1])
print(files[1])
print(tbl[1].get_colnames())
print(tbl[1].get_nrows())

/global/cfs/projectdirs/desi/survey/catalogs/Y3/LSS/loa-v1/LSScats/v1.1/nonKP/BGS_ANY_NGC_clustering.dat.fits
['TARGETID', 'Z', 'NTILE', 'RA', 'DEC', 'PHOTSYS', 'FRAC_TLOBS_TILES', 'WEIGHT_ZFAIL', 'WEIGHT', 'WEIGHT_COMP', 'WEIGHT_SYS', 'flux_g_dered', 'flux_r_dered', 'flux_z_dered', 'flux_w1_dered', 'flux_w2_dered', 'NX', 'WEIGHT_FKP']
8052629


In [2]:
import numpy as np
np.linspace(0.03, 3, 51)

array([0.03  , 0.0894, 0.1488, 0.2082, 0.2676, 0.327 , 0.3864, 0.4458,
       0.5052, 0.5646, 0.624 , 0.6834, 0.7428, 0.8022, 0.8616, 0.921 ,
       0.9804, 1.0398, 1.0992, 1.1586, 1.218 , 1.2774, 1.3368, 1.3962,
       1.4556, 1.515 , 1.5744, 1.6338, 1.6932, 1.7526, 1.812 , 1.8714,
       1.9308, 1.9902, 2.0496, 2.109 , 2.1684, 2.2278, 2.2872, 2.3466,
       2.406 , 2.4654, 2.5248, 2.5842, 2.6436, 2.703 , 2.7624, 2.8218,
       2.8812, 2.9406, 3.    ])